In [1]:
# Système
import os
import sys
import time

# Built-in imports
import warnings
from collections import Counter
import pickle

# Data manipulation
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
import missingno

# Sklearn imports
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    root_mean_squared_error,
    mean_absolute_error
)

# ML Models - Linear
from sklearn.linear_model import (
    LogisticRegression,
    Perceptron,
    SGDClassifier,
    Lasso,
    LassoCV
)


from sklearn.preprocessing import LabelEncoder

# ML Models - SVM
from sklearn.svm import SVC, LinearSVC

# ML Models - Ensemble
from sklearn.ensemble import RandomForestClassifier

# ML Models - Other
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

# Disable warnings
warnings.filterwarnings('ignore')

In [2]:
from google.colab import drive
drive.mount("/content/drive/", force_remount=True)

Mounted at /content/drive/


# MPI TRAIN AND STORE THE BEST MODEL

In [3]:
doc1_df_train = pd.read_csv("/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1MPI/doc1/doc1_df_train.csv");
doc2_df_train = pd.read_csv("/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1MPI/doc2/doc2_df_train.csv");
doc3_df_train = pd.read_csv("/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1MPI/doc3/doc3_df_train.csv");

In [4]:
doc1_df_train

,REGION_DE_NAISSANCE,CREDIT,NIVEAU,SESSION,MENTION,MOYENNE ANNUELLE,RESULTAT,RESULTAT APP EVALUATION,Année BAC,Sexe,...,RESULTAT_Encode,SESSION_Encode,MENTION_Encode,Academie perf.,Residence perf.,Résidence_Encode,Ets. de provenance_Encode,Centre d'Ec._Encode,Académie de l'Ets. Prov._Encode,REGION_DE_NAISSANCE_Encode
0,Dakar,60,1,Première Session,Passable,11.61,PASSE,1,2018,M,...,2,1,0,10.668400,10.627500,70,95,72,10,0
1,Thiès,15,1,Deuxième Session,NaN,3.34,NON ADMIS,1,2018,M,...,0,0,4,10.926122,11.295625,76,33,18,14,9
2,Dakar,60,1,Première Session,Assez-Bien,13.43,PASSE,1,2018,M,...,2,1,1,10.693385,10.722727,42,75,54,9,0
3,Kaolack,43,1,Deuxième Session,NaN,8.90,AUTORISE,1,2018,M,...,1,0,4,10.863333,11.245000,29,139,12,4,3
4,Tambacounda,45,1,Deuxième Session,NaN,7.89,AUTORISE,1,2018,M,...,1,0,4,11.000000,11.202000,73,123,89,13,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
273,Dakar,20,1,Deuxième Session,NaN,4.83,NON ADMIS,1,2018,M,...,0,0,4,10.693385,10.695217,60,100,10,9,0
274,Dakar,60,1,Deuxième Session,Passable,10.24,PASSE,1,2018,F,...,2,0,0,11.080526,10.754516,7,138,104,0,0
275,0,52,1,Deuxième Session,NaN,10.06,AUTORISE,1,2018,M,...,1,0,4,11.080526,12.524286,57,131,95,0,11
276,Louga,60,1,Première Session,Passable,10.21,PASSE,1,2018,M,...,2,1,0,10.847500,10.840000,30,120,87,7,5


In [5]:
doc1_df_train.dtypes

,0
REGION_DE_NAISSANCE,object
CREDIT,int64
NIVEAU,int64
SESSION,object
MENTION,object
MOYENNE ANNUELLE,float64
RESULTAT,object
RESULTAT APP EVALUATION,int64
Année BAC,int64
Sexe,object


In [6]:
columns = ["MATH", "SCPH", "SVT", "FR", "PHILO", "AN", "Moy. Gle",
           "Moy. sur Mat.Fond.", "Age en Décembre 2018", "S1", "S2",
           "Academie perf.", "Residence perf."]

from itertools import combinations

fixedSize = 6

# Obtenir uniquement les combinaisons de taille fixedSize
res = list(combinations(columns, fixedSize))

print(f"Nombre de combinaisons de {fixedSize} éléments : {len(res)}")

Nombre de combinaisons de 6 éléments : 1716


In [7]:
from math import sqrt
import pandas as pd

def compute_distance_errors(df, col1, col2):
    """
    Calcule les erreurs MAE et RMSE entre deux colonnes d'un DataFrame,
    ajoute les colonnes correspondantes et renvoie les moyennes des erreurs.

    Args:
        df (pd.DataFrame): Le DataFrame contenant les colonnes.
        col1 (str): Le nom de la première colonne.
        col2 (str): Le nom de la seconde colonne.

    Returns:
        tuple: (mae_distance_error, rmse_distance_error)
    """
    # Calcul des erreurs élémentaires
    df['mae_distance_error'] = abs(df[col1] - df[col2])
    df['rmse_distance_error'] = (df[col1] - df[col2])**2

    # Moyennes des erreurs
    mae_distance_error = df['mae_distance_error'].mean()
    rmse_distance_error = sqrt(df['rmse_distance_error'].mean())

    # Retour des erreurs
    return mae_distance_error, rmse_distance_error

In [8]:
def root_mean_squared_error(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [9]:
def adjust_predictions(y_pred, y_true):
   """
   Ajuste les prédictions pour les maintenir dans les limites réalistes des données observées
   et arrondit les résultats.

   Args:
       y_pred (array-like): Les prédictions brutes du modèle
       y_true (array-like): Les vraies valeurs observées

   Returns:
       array-like: Les prédictions ajustées

   Example:
       >>> y_true = [12, 15, 8, 17, 10]
       >>> y_pred = [13.456, 22.789, -3.123, 16.789, 9.234]
       >>> adjusted = adjust_predictions(y_pred, y_true)
       >>> print(adjusted)
       [13.46, 17.00, 8.00, 16.79, 9.23]
   """
   min_score = min(y_true)
   max_score = max(y_true)
   y_pred = np.clip(y_pred, min_score, max_score)
   y_pred = np.round(y_pred, decimals=2)
   return y_pred

In [10]:
def calculate_mrr(y_true, y_pred):
    """
    Calcule le Mean Reciprocal Rank (MRR).

    Args:
        y_true (pd.Series): Valeurs réelles (avec des indices correspondant à y_pred).
        y_pred (np.ndarray): Prédictions associées aux valeurs réelles.

    Returns:
        float: MRR calculé.
    """
    sorted_indices = np.argsort(-y_pred)  # Indices triés par prédictions descendantes
    ranks = np.argsort(sorted_indices) + 1  # Rangs associés aux prédictions
    reciprocal_ranks = 1 / ranks[y_true.index]
    return sum(reciprocal_ranks)

In [11]:
def train_lasso_models(train_dfs, res, names=['Doc1', 'Doc2', 'Doc3'], target_col='Score L1', alpha=0.01):
    """
    Entraîne les modèles Lasso sur les données d'entraînement avec validation.

    Args:
        train_dfs (list): Liste des DataFrames d'entraînement.
        res (list): Liste des combinaisons de features à tester.
        names (list): Noms des documents/modèles.
        target_col (str): Nom de la colonne cible.
        alpha (float): Paramètre de régularisation Lasso.

    Returns:
        dict: Dictionnaire contenant les meilleurs modèles pour chaque document.
    """
    best_models = {}
    # val_best_rmse = None
    # val_best_mae = None

    for train_df, name in zip(train_dfs, names):
        print(f"\nEntraînement pour {name}")
        print("=" * 50)

        validation_results = []
        train_data, val_data = train_test_split(train_df, test_size=0.2, random_state=42)

        for feature_cols in res:
            # Préparation des données
            X_train = train_data[list(feature_cols)]
            y_train = train_data[target_col]
            X_val = val_data[list(feature_cols)]
            y_val = val_data[target_col]

            # Standardisation
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_val_scaled = scaler.transform(X_val)

            # Entraînement
            lasso = Lasso(alpha=alpha, positive=True)
            lasso.fit(X_train_scaled, y_train)

            # Prédictions et métriques
            y_val_pred = lasso.predict(X_val_scaled)
            val_rmse = ((y_val - y_val_pred) ** 2).mean() ** 0.5
            val_mae = abs(y_val - y_val_pred).mean()

            # val_best_rmse = val_rmse
            # val_best_mae = val_mae

            validation_results.append({
                'features': feature_cols,
                'model': lasso,
                'scaler': scaler,
                'coefs': lasso.coef_.ravel(),
                'val_rmse': val_rmse,
                'val_mae': val_mae
            })

            # print(f"Validation RMSE: {val_rmse:.4f}, MAE: {val_mae:.4f}")


            # Afficher les coefficients LASSO
            feature_cols_selected = feature_cols[:len(lasso.coef_)]
            lasso_coefficients_df = pd.DataFrame({'Feature': feature_cols_selected, 'Coefficient': lasso.coef_.ravel()})
            lasso_coefficients_df = lasso_coefficients_df.sort_values(by='Coefficient', ascending=False)
            print("Coefficients LASSO :\n", lasso_coefficients_df)
            print("RMSE :", val_rmse)
            print("MAE :", val_mae, "\n", ("_"*30))


        # Sélection du meilleur modèle
        best_val_model = min(validation_results, key=lambda x: x['val_rmse'])
        best_models[name] = best_val_model

    return best_models

In [12]:
def save_models(final_results, base_path='models'):
    """
    Sauvegarde les modèles et leurs résultats.

    Args:
        final_results (dict): Résultats finaux contenant les modèles.
        base_path (str): Chemin de base pour la sauvegarde.
    """
    os.makedirs(base_path, exist_ok=True)

    for name, results in final_results.items():
        model_path = os.path.join(base_path, name)
        os.makedirs(model_path, exist_ok=True)

        # Sauvegarder le modèle
        with open(os.path.join(model_path,'lasso_globale_model.pkl'), 'wb') as f:
            pickle.dump(results['model'], f)

        # Sauvegarder le scaler
        with open(os.path.join(model_path, 'lasso_globale_scaler.pkl'), 'wb') as f:
            pickle.dump(results['scaler'], f)

        # Sauvegarder les autres informations
        info = {
            'val_rmse': results['val_rmse'],
            'val_mae': results['val_mae'],
            'features': results['features']
        }
        with open(os.path.join(model_path, 'lasso_globale_info.pkl'), 'wb') as f:
            pickle.dump(info, f)

        print(f"Modèle {name} sauvegardé dans {model_path}")

In [13]:
# Préparer les données
train_dfs = [doc1_df_train, doc2_df_train, doc3_df_train]
names = ['Doc1', 'Doc2', 'Doc3']

# Entraîner les modèles
best_models = train_lasso_models(
    train_dfs=train_dfs,  # Liste des DataFrames d'entraînement
    res=res,             # Combinaisons de features à tester
    names=names,         # Noms des documents
    target_col='Score L1', # Variable cible
    alpha=0.01           # Paramètre de régularisation
)


# Sauvegarder les modèles
save_models(
    final_results=best_models,
    base_path='models'  # Dossier de sauvegarde
)

# print("Best Model : ", best_models)


# Reformater les données
data = []
for doc, values in best_models.items():
    data.append({
        'Document': doc,
        'Features': ', '.join(values['features']),
        # 'Modèle': str(values['model']),
        # 'Scaler': str(values['scaler']),
        'coefs': [round(float(c), 2) for c in values['coefs']],
        'RMSE Validation': round(float(values['val_rmse']), 2),
        'MAE Validation': round(float(values['val_mae']), 2)
    })

# Créer un DataFrame
df_best_models = pd.DataFrame(data)

# Afficher joliment
print(df_best_models.to_string(index=False))


Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
               Feature  Coefficient
3                  AN    13.238239
4  Moy. sur Mat.Fond.    11.165999
2               PHILO     8.864715
0                 SVT     2.209193
1                  FR     1.466122
5                  S2     0.000000
RMSE : 42.01828490541567
MAE : 36.82078627944441 
 ______________________________
Coefficients LASSO :
               Feature  Coefficient
3                  AN    13.233608
4  Moy. sur Mat.Fond.    11.135414
2               PHILO     8.859768
0                 SVT     2.208416
1                  FR     1.451881
5      Academie perf.     0.109656
RMSE : 42.032276642193146
MAE : 36.8506972079422 
 ______________________________
Coefficients LASSO :
               Feature  Coefficient
3                  AN    13.238239
4  Moy. sur Mat.Fond.    11.165999
2               PHILO     8.864715
0                 SVT     2.209193
1                  FR     1.466122
5     Residen

In [14]:
df_best_models

,Document,Features,coefs,RMSE Validation,MAE Validation
0,Doc1,"MATH, FR, PHILO, Age en Décembre 2018, S1, S2","[13.38, 0.0, 5.46, 0.0, 28.7, 0.0]",36.86,27.66
1,Doc2,"MATH, SCPH, SVT, PHILO, Moy. Gle, S1","[9.1, 9.06, 0.0, 3.61, 2.72, 26.36]",40.46,32.38
2,Doc3,"MATH, SCPH, FR, AN, Moy. sur Mat.Fond., S1","[12.72, 6.59, 0.0, 4.02, 1.54, 30.01]",35.44,28.95


# PCSM TRAIN AND STORE THE BEST MODEL

In [15]:
doc1_df_train = pd.read_csv("/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1PCSM/doc1/doc1_df_train.csv");
doc2_df_train = pd.read_csv("/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1PCSM/doc2/doc2_df_train.csv");
doc3_df_train = pd.read_csv("/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1PCSM/doc3/doc3_df_train.csv");

In [16]:
# Préparer les données
train_dfs = [doc1_df_train, doc2_df_train, doc3_df_train]
names = ['Doc1', 'Doc2', 'Doc3']

# Entraîner les modèles
best_models = train_lasso_models(
    train_dfs=train_dfs,  # Liste des DataFrames d'entraînement
    res=res,             # Combinaisons de features à tester
    names=names,         # Noms des documents
    target_col='Score L1', # Variable cible
    alpha=0.01           # Paramètre de régularisation
)


# Sauvegarder les modèles
save_models(
    final_results=best_models,
    base_path='models'  # Dossier de sauvegarde
)

# print("Best Model : ", best_models)


# Reformater les données
data = []
for doc, values in best_models.items():
    data.append({
        'Document': doc,
        'Features': ', '.join(values['features']),
        # 'Modèle': str(values['model']),
        # 'Scaler': str(values['scaler']),
        'coefs': [round(float(c), 2) for c in values['coefs']],
        'RMSE Validation': round(float(values['val_rmse']), 2),
        'MAE Validation': round(float(values['val_mae']), 2)
    })

# Créer un DataFrame
df_best_models = pd.DataFrame(data)

# Afficher joliment
print(df_best_models.to_string(index=False))

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
               Feature  Coefficient
4  Moy. sur Mat.Fond.     3.436728
3                  AN     2.593365
0                 SVT     0.916770
1                  FR     0.389221
2               PHILO     0.000000
5                  S2     0.000000
RMSE : 33.82409202585069
MAE : 19.555356247720745 
 ______________________________
Coefficients LASSO :
               Feature  Coefficient
4  Moy. sur Mat.Fond.     3.190951
3                  AN     2.577091
5      Academie perf.     1.634633
0                 SVT     0.948628
1                  FR     0.252187
2               PHILO     0.000000
RMSE : 34.05267173887877
MAE : 19.493681412144014 
 ______________________________
Coefficients LASSO :
               Feature  Coefficient
4  Moy. sur Mat.Fond.     3.436728
3                  AN     2.593365
0                 SVT     0.916770
1                  FR     0.389221
2               PHILO     0.000000
5     Resid

# BCGS TRAIN AND STORE THE BEST MODEL

In [17]:
doc1_df_train = pd.read_csv("/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1BCGS/doc1/doc1_df_train.csv");
doc2_df_train = pd.read_csv("/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1BCGS/doc2/doc2_df_train.csv");
doc3_df_train = pd.read_csv("/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1BCGS/doc3/doc3_df_train.csv");

In [18]:
# Préparer les données
train_dfs = [doc1_df_train, doc2_df_train, doc3_df_train]
names = ['Doc1', 'Doc2', 'Doc3']

# Entraîner les modèles
best_models = train_lasso_models(
    train_dfs=train_dfs,  # Liste des DataFrames d'entraînement
    res=res,             # Combinaisons de features à tester
    names=names,         # Noms des documents
    target_col='Score L1', # Variable cible
    alpha=0.01           # Paramètre de régularisation
)


# Sauvegarder les modèles
save_models(
    final_results=best_models,
    base_path='models'  # Dossier de sauvegarde
)

# print("Best Model : ", best_models)


# Reformater les données
data = []
for doc, values in best_models.items():
    data.append({
        'Document': doc,
        'Features': ', '.join(values['features']),
        # 'Modèle': str(values['model']),
        # 'Scaler': str(values['scaler']),
        'coefs': [round(float(c), 2) for c in values['coefs']],
        'RMSE Validation': round(float(values['val_rmse']), 2),
        'MAE Validation': round(float(values['val_mae']), 2)
    })

# Créer un DataFrame
df_best_models = pd.DataFrame(data)

# Afficher joliment
print(df_best_models.to_string(index=False))

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
               Feature  Coefficient
1                  FR     1.914619
2               PHILO     1.237163
0                 SVT     0.000000
3                  AN     0.000000
4  Moy. sur Mat.Fond.     0.000000
5                  S2     0.000000
RMSE : 39.75436549770721
MAE : 32.97448748336353 
 ______________________________
Coefficients LASSO :
               Feature  Coefficient
1                  FR     1.914619
2               PHILO     1.237163
0                 SVT     0.000000
3                  AN     0.000000
4  Moy. sur Mat.Fond.     0.000000
5      Academie perf.     0.000000
RMSE : 39.75436549770721
MAE : 32.97448748336353 
 ______________________________
Coefficients LASSO :
               Feature  Coefficient
1                  FR     1.883051
2               PHILO     1.200935
5     Residence perf.     0.522900
0                 SVT     0.000000
3                  AN     0.000000
4  Moy. sur M